# Work Script: apply_review
Converted from `sft/work/apply_review.py`.


In [ ]:
from __future__ import annotations


In [ ]:
"""
sft/work/apply_review.py — apply human review decisions to behavior and probe sets (B5)

This script is run by the HUMAN reviewer after filling in behavior_review.csv and
probes_review.csv. It is the ONLY mechanism that may change review_status from
"draft" to "approved" or "rejected".

Usage
-----
python sft/work/apply_review.py \\
    --behavior-jsonl  sft/work/behavior_dialogues.jsonl \\
    --behavior-review sft/work/behavior_review.csv \\
    --probes-jsonl    sft/work/probes_all.jsonl \\
    --probes-review   sft/work/probes_review.csv \\
    --out-behavior    sft/work/behavior_dialogues_approved.jsonl \\
    --out-probes      sft/work/probes_approved.jsonl

After running, the caller should rebuild probes_final.jsonl from probes_approved.jsonl
(final-partition items only).
"""

import argparse
import csv
import json
import sys
from pathlib import Path
from typing import Any


VALID_DECISIONS = {'ok', 'fix', 'reject'}

In [ ]:
def load_jsonl(path: Path) -> dict[str, dict]:
    items = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            items[item['id']] = item
    return items

In [ ]:
def load_review_csv(path: Path) -> dict[str, dict[str, str]]:
    """Returns {id: {decision, notes}}."""
    decisions = {}
    with open(path, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            row_id = row.get('id', '').strip()
            dec    = row.get('decision', '').strip().lower()
            notes  = row.get('notes', '').strip()
            if not row_id:
                continue
            if dec not in VALID_DECISIONS and dec != '':
                print(f'WARNING: unknown decision {dec!r} for id {row_id!r} — treating as empty (no change)',
                      file=sys.stderr)
                dec = ''
            decisions[row_id] = {'decision': dec, 'notes': notes}
    return decisions

In [ ]:
def apply_decisions(
    items: dict[str, dict],
    decisions: dict[str, dict[str, str]],
    item_type: str,
) -> tuple[list[dict], dict[str, int]]:
    """
    Apply review decisions to items.

    Decision mapping:
      'ok'     → review_status = 'approved'
      'reject' → excluded from output
      'fix'    → review_status = 'needs_fix' (kept with flag; human must re-review)
      ''       → review_status unchanged ('draft')

    Returns (output_list, counts).
    """
    counts = {'approved': 0, 'rejected': 0, 'needs_fix': 0, 'draft': 0}
    output = []

    for item_id, item in sorted(items.items()):
        dec_info = decisions.get(item_id, {'decision': '', 'notes': ''})
        dec      = dec_info['decision']
        notes    = dec_info['notes']

        item = dict(item)  # shallow copy
        if notes:
            item['review_notes'] = notes

        if dec == 'ok':
            item['review_status'] = 'approved'
            counts['approved'] += 1
            output.append(item)
        elif dec == 'reject':
            item['review_status'] = 'rejected'
            counts['rejected'] += 1
            # excluded from output
        elif dec == 'fix':
            item['review_status'] = 'needs_fix'
            counts['needs_fix'] += 1
            output.append(item)
        else:
            # unchanged
            counts[item.get('review_status', 'draft')] = \
                counts.get(item.get('review_status', 'draft'), 0) + 1
            output.append(item)

    return output, counts

In [ ]:
def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description='Apply human review to behavior/probe sets.')
    parser.add_argument('--behavior-jsonl',  required=True)
    parser.add_argument('--behavior-review', required=True)
    parser.add_argument('--probes-jsonl',    required=True)
    parser.add_argument('--probes-review',   required=True)
    parser.add_argument('--out-behavior',    required=True)
    parser.add_argument('--out-probes',      required=True)
    parser.add_argument('--rebuild-final',   action='store_true',
                        help='Also write probes_final.jsonl from the final partition of approved probes')
    parser.add_argument('--out-probes-final', default=None)
    args = parser.parse_args(argv)

    # Load
    beh_items  = load_jsonl(Path(args.behavior_jsonl))
    beh_review = load_review_csv(Path(args.behavior_review))
    prb_items  = load_jsonl(Path(args.probes_jsonl))
    prb_review = load_review_csv(Path(args.probes_review))

    # Apply
    beh_out, beh_counts = apply_decisions(beh_items, beh_review, 'behavior')
    prb_out, prb_counts = apply_decisions(prb_items, prb_review, 'probe')

    # Write behavior
    out_beh = Path(args.out_behavior)
    out_beh.parent.mkdir(parents=True, exist_ok=True)
    with open(out_beh, 'w', encoding='utf-8', newline='\n') as f:
        for item in beh_out:
            f.write(json.dumps(item, ensure_ascii=False, sort_keys=True) + '\n')

    # Write probes
    out_prb = Path(args.out_probes)
    out_prb.parent.mkdir(parents=True, exist_ok=True)
    with open(out_prb, 'w', encoding='utf-8', newline='\n') as f:
        for item in prb_out:
            f.write(json.dumps(item, ensure_ascii=False, sort_keys=True) + '\n')

    # Optionally rebuild probes_final.jsonl
    if args.rebuild_final:
        if not args.out_probes_final:
            print('ERROR: --out-probes-final required with --rebuild-final', file=sys.stderr)
            return 2
        final_items = [p for p in prb_out
                       if p.get('partition') == 'final' and p.get('review_status') == 'approved']
        out_final = Path(args.out_probes_final)
        with open(out_final, 'w', encoding='utf-8', newline='\n') as f:
            for item in final_items:
                f.write(json.dumps(item, ensure_ascii=False, sort_keys=True) + '\n')
        print(f'probes_final.jsonl: {len(final_items)} items → {out_final}')

    # Report
    print(f'Behavior: {beh_counts}  → {out_beh}  ({len(beh_out)} records)')
    print(f'Probes:   {prb_counts}  → {out_prb}  ({len(prb_out)} records)')
    return 0

In [ ]:
print("RESULT: PASS apply_review.py notebook compiled successfully")